In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate bitsandbytes

In [ ]:
from google.colab import userdata
import os

# Lee la clave desde los Secrets de Colab (icono de llave en la barra lateral)
os.environ["HF_TOKEN"] = userdata.get("KEY_HF")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # liviano para Colab gratis

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

In [ ]:
knowledge_base = [
    { "reclamo": "No me llegó el producto que compré hace una semana","respuesta": "Lamentamos la demora. Por favor envíanos tu número de pedido y verificaremos el estado de envío en un plazo de 24 horas."},
    { "reclamo": "El producto llegó dañado", "respuesta": "Sentimos mucho lo ocurrido. Adjunta una foto del producto y del empaque, y procederemos con el reemplazo o reembolso en 3-5 días hábiles." },
    { "reclamo": "El producto llegó dañado","respuesta": "Sentimos mucho lo ocurrido. Adjunta una foto del producto y procederemos con el reemplazo o reembolso en 3-5 días hábiles."},
    { "reclamo": "Quiero cancelar mi suscripción","respuesta": "Puedes cancelar desde tu perfil en Configuración > Suscripciones. Si necesitas ayuda, indícanos tu correo registrado."},
    { "reclamo": "El producto llegó dañado", "respuesta": "Sentimos mucho lo ocurrido. Adjunta una foto del producto y del empaque, y procederemos con el reemplazo o reembolso en 3-5 días hábiles." },
    { "reclamo": "Me llegó roto", "respuesta": "Lamentamos el inconveniente. Por favor envíanos fotos del artículo dañado y lo gestionamos de inmediato con un reemplazo o reembolso." },
    { "reclamo": "El artículo está defectuoso", "respuesta": "Pedimos disculpas por esto. Comparte imágenes del defecto y coordinaremos el cambio o devolución del dinero en un plazo de 3-5 días hábiles." },
    { "reclamo": "Recibí un producto en mal estado", "respuesta": "Lo sentimos mucho. Necesitamos que nos envíes fotos del producto y del embalaje para iniciar el proceso de reemplazo o reembolso." },
    { "reclamo": "El paquete llegó aplastado y el producto dentro está destrozado", "respuesta": "Entendemos tu frustración y pedimos disculpas. Envíanos fotos del paquete y del producto y resolveremos con reemplazo o reembolso en 3-5 días hábiles." },
    { "reclamo": "El producto tiene una grieta", "respuesta": "Sentimos que hayas recibido el producto en ese estado. Por favor adjunta una fotografía de la grieta y procesaremos el reemplazo o reembolso a la brevedad." },
    { "reclamo": "Llegó con partes rotas", "respuesta": "Lamentamos el inconveniente. Envíanos imágenes de las partes afectadas y gestionaremos el reemplazo o reembolso en 3-5 días hábiles." },
    { "reclamo": "El producto no funciona, creo que llegó fallado de fábrica", "respuesta": "Disculpa los inconvenientes. Cuéntanos el fallo que presenta y adjunta fotos o video si puedes, para validar si es un defecto de fábrica y proceder con el cambio." },
    { "reclamo": "Me enviaron un producto que claramente está usado o dañado", "respuesta": "Esto no debería haber ocurrido y lo sentimos mucho. Por favor adjunta fotos del estado del producto y gestionaremos un reemplazo o reembolso de forma prioritaria." },
    { "reclamo": "El producto dejó de funcionar al primer uso", "respuesta": "Lamentamos que hayas tenido esta experiencia. Comparte fotos o video del inconveniente y revisaremos si aplica garantía para reemplazo o reembolso." },
    { "reclamo": "Tiene rayones y golpes, llegó en pésimas condiciones", "respuesta": "Pedimos disculpas por el estado en que llegó tu pedido. Envíanos fotografías del producto y del empaque y procederemos con el reemplazo o reembolso." },
    { "reclamo": "El producto está incompleto, le faltan piezas", "respuesta": "Lo sentimos. Indícanos qué piezas faltan y adjunta fotos del contenido recibido para enviarte las piezas faltantes o gestionar un reemplazo completo." },
    { "reclamo": "Llegó con manchas y suciedad, no parece nuevo", "respuesta": "Entendemos tu molestia y pedimos disculpas. Envíanos imágenes del producto para verificar y proceder con el reemplazo o reembolso correspondiente." },
    { "reclamo": "El producto se rompió apenas lo abrí", "respuesta": "Sentimos mucho lo sucedido. Adjunta fotos del producto y del empaque original para que podamos gestionar el reemplazo o reembolso en el menor tiempo posible." },
    { "reclamo": "Recibí el producto con el empaque violado y el artículo dañado", "respuesta": "Lamentamos esta situación. Por favor envíanos imágenes del empaque y del producto afectado para iniciar de inmediato el proceso de reemplazo o reembolso." },
    { "reclamo": "La pantalla llegó quebrada", "respuesta": "Pedimos disculpas por este inconveniente. Comparte fotos claras del daño y coordinaremos el reemplazo o reembolso en 3-5 días hábiles." },
    { "reclamo": "El electrodoméstico no enciende, llegó sin funcionar", "respuesta": "Sentimos que hayas tenido este problema. Adjunta fotos y, de ser posible, un video intentando encenderlo, para evaluar si aplica garantía y proceder con el cambio." },
    { "reclamo": "Me mandaron el producto equivocado y además está dañado", "respuesta": "Doble inconveniente y lo sentimos mucho. Envíanos fotos del producto recibido y del empaque, y gestionaremos el envío del artículo correcto o el reembolso." },
    { "reclamo": "El juguete que compré para mi hijo llegó roto", "respuesta": "Sentimos mucho que esto haya pasado. Por favor adjunta fotos del juguete y del empaque y lo reemplazaremos o reembolsaremos en 3-5 días hábiles." },
    { "reclamo": "El material del producto es de muy mala calidad, se desarmó solo", "respuesta": "Lamentamos que el producto no haya cumplido tus expectativas de calidad. Comparte fotos o video del desperfecto para evaluar si aplica reemplazo o reembolso bajo garantía." },
]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Convertir reclamos a vectores
reclamos = [item["reclamo"] for item in knowledge_base]
embeddings = embedder.encode(reclamos, convert_to_numpy=True)

# Crear índice FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [ ]:
def responder_reclamo(pregunta, top_k=2):
    # 1. Buscar casos similares
    query_vec = embedder.encode([pregunta], convert_to_numpy=True)
    distancias, indices = index.search(query_vec, top_k)

    contexto = "\n\n".join([
        f"Reclamo previo: {knowledge_base[i]['reclamo']}\nRespuesta: {knowledge_base[i]['respuesta']}"
        for i in indices[0]
    ])

    # 2. Construir prompt para Qwen
    messages = [
        {"role": "system", "content": "Eres un agente de atención al cliente. Responde el reclamo del usuario usando el mismo estilo y tono de las respuestas previas del equipo."},
        {"role": "user", "content": f"Casos similares anteriores:\n{contexto}\n\nNuevo reclamo: {pregunta}\n\nRespuesta:"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    respuesta = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return respuesta

In [ ]:
print(responder_reclamo("Mi pedido no ha llegado y ya pasaron 16 dias"))